In [ ]:
# Code Block 1: Notebook description
#Notebook description

# This notebook evaluates the performance of a portfolio of assets on a buy-and-hold basis.
# It focuses on how multiple assets interact within the portfolio, rather than assessing a specific mechanical trading strategy.
# buy and hold is a good benchmark for any trading strategy, so it is important to evaluate the performance of a portfolio independently of 
# trading strategies we may want to implement on the individual assets.


In [ ]:
# Code Block 2: Load Libraries

# Load Libraries

import numpy as np

import pandas as pd



import statsmodels

import statsmodels.api as sm

from statsmodels.tsa.stattools import coint

from IPython.display import display

import schwab

from schwab.auth import easy_client

from schwab.client import Client

import re 

import os

import matplotlib.pyplot as plt

import plotly.express as px

from plotly.subplots import make_subplots

import plotly.graph_objects as go

import sys

from pathlib import Path



def find_project_root(start: Path) -> Path:

    for candidate in (start, *start.parents):

        if (candidate / "pyproject.toml").exists() and (candidate / "Quantapp").is_dir():

            return candidate

    raise RuntimeError("Could not locate the project root containing the Quantapp package.")



PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(PROJECT_ROOT))



import yfinance as yf

#from Quantapp.analytics import TimeSeriesAnalytics as Rolling

from Quantapp.data import MacroDataClient

from Quantapp.secrets import load_project_env, require_secret



load_project_env()



#qc = Rolling()

qe = MacroDataClient()

In [ ]:
# Code Block 3: Define functions and classes
#Define functions & Classes


#takes a dict of portfolio and their total amounts and directional (short or long value), converts dict to weightings instead of absolute values
def create_weighted_portfolio(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: (amount / total) * (1 if direction == 'long' else -1)
            for (ticker, amount), direction in zip(portfolio.items(), ['long' if amount >= 0 else 'short' for amount in portfolio.values()])
    }

def create_weight_dict(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: amount / total for ticker, amount in portfolio.items()}

def create_equal_weighted_dict(tickers):
    n = len(tickers)
    if n == 0:
        return {}
    equal_weight = 1 / n
    return {ticker: equal_weight for ticker in tickers}

def normalize_yf_ticker(ticker):
    if not isinstance(ticker, str):
        return ticker
    return ticker.strip().replace('/', '-')

def build_yf_ticker_map(tickers):
    return {ticker: normalize_yf_ticker(ticker) for ticker in tickers}

def zscore_series(series):
    mean = series.mean()
    std = series.std(ddof=0)
    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)
    return (series - mean) / std

def z_score(series):
    mean = series.mean()
    std  = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)

    z = (series - mean) / std
    return z.replace([np.inf, -np.inf], np.nan)

def sharpe_annualized(series):
    mean = series.mean()
    std  = series.std(ddof=0)
    if std == 0 or np.isnan(std):
        return 0.0
    return (mean / std) * np.sqrt(252)


class Portfolio:
    def __init__(self, ticker_dict ,period='10y', interval='1d', equal_weighted=False, portfolio_name='Portfolio'):
        self.portfolio_name = portfolio_name
        self.period = period
        self.interval = interval
        if equal_weighted:
            self.ticker_dict = create_equal_weighted_dict(list(ticker_dict.keys()))
        else:
            self.ticker_dict = ticker_dict
        self.tickers = list(ticker_dict.keys())
        self.yf_ticker_map = build_yf_ticker_map(self.tickers)
        self.yf_tickers = [self.yf_ticker_map[ticker] for ticker in self.tickers]
        self.data = self.get_data() #close prices of the tickers
        
    def __name__(self):
        return self.portfolio_name
    
    def create_equal_weighted_dict(self):
        return {ticker: 1/len(self.tickers) for ticker in self.tickers}
    
    def rebalance_to_equal_weighted(self):
        self.ticker_dict = self.create_equal_weighted_dict()
        
    def _restore_original_ticker_names(self, close_data):
        if isinstance(close_data, pd.Series):
            close_data = close_data.copy()
            close_data.name = self.tickers[0]
            return close_data
        rename_map = {yf_ticker: ticker for ticker, yf_ticker in self.yf_ticker_map.items()}
        return close_data.rename(columns=rename_map)

    #retrieve raw prices using yf.Ticker as a dataframe
    def retrieve_raw_prices(self, data_type='dataframe'):
        """
        Retrieves historical Close price data for all tickers in ticker_dict using yfinance.

        Returns:
            pd.DataFrame: A DataFrame containing Close prices with dates as index and tickers as columns.
        """
        # Download historical data for all tickers
        data = yf.download(
            tickers=self.yf_tickers,
            period=self.period,
            interval=self.interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )
        
        close_data = self._restore_original_ticker_names(data['Close'])

        if data_type == 'dataframe':
            return close_data
        elif data_type == 'dict':
            if isinstance(close_data, pd.Series):
                return {self.tickers[0]: close_data}
            return {ticker: close_data[ticker] for ticker in self.tickers}
        else:
            raise ValueError('data_type should be either dataframe or dict')

    def get_data(self):
        """
        Retrieves historical Close price data for all tickers in ticker_dict using yfinance.

        Returns:
            pd.DataFrame: A DataFrame containing Close prices with dates as index and tickers as columns.
        """
        # Download historical data for all tickers
        data = yf.download(
            tickers=self.yf_tickers,
            period=self.period,
            interval=self.interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )
        
        return self._restore_original_ticker_names(data['Close'])

    def returns(self, n=1, return_type='portfolio'):
        """
        Calculates the daily returns of the portfolio or individual assets.

        Args:
            n (int): The number of periods over which to calculate returns.
            allocation_type (str): Type of returns to calculate.
                                    - 'portfolio' (default): Returns the portfolio's daily returns.
                                    - 'individual': Returns a DataFrame of each individual asset's daily returns.

        Returns:
            pd.Series or pd.DataFrame: 
                - If allocation_type is 'portfolio', returns a Series of portfolio returns.
                - If allocation_type is 'individual', returns a DataFrame of individual asset returns.
        """
        # Calculate daily percentage change
        daily_returns = self.data.pct_change(n).dropna(how='all')

        if return_type == 'portfolio':
            # Compute portfolio returns by multiplying asset returns with their respective weights
            # and summing them up
            portfolio_returns = daily_returns.dot(list(self.ticker_dict.values()))
            return portfolio_returns
        elif return_type == 'individual':
            # Return individual asset returns as a DataFrame
            return daily_returns
        else:
            raise ValueError("Invalid return_type. Choose 'portfolio' or 'individual'.")
           
    def cumulative_returns(self, n=200, return_type='portfolio'):
        """
        Calculates the cumulative returns of the portfolio or individual assets over the last n days.

        Args:
            n (int): Number of recent days to calculate cumulative returns.
            allocation_type (str): Type of returns to calculate.
                                    - 'portfolio' (default): Returns cumulative returns of the portfolio.
                                    - 'individual': Returns cumulative returns of individual assets.

        Returns:
            pd.Series or pd.DataFrame: 
                - If 'portfolio', returns a Series of cumulative returns.
                - If 'individual', returns a DataFrame of cumulative returns for each asset.
        """

        # Retrieve the returns based on allocation type
        all_returns = self.returns(n=n, return_type=return_type)
        
        # Ensure there are enough data points
        if return_type == 'portfolio':
            if len(all_returns) < n:
                raise ValueError(f"Not enough data to calculate cumulative returns for the last {n} days.")
        elif return_type == 'individual':
            if len(all_returns) < n:
                raise ValueError(f"Not enough data to calculate cumulative returns for the last {n} days.")
        else:
            raise ValueError("Invalid return_type. Choose 'portfolio' or 'individual'.")
        
        # Slice the last n days of returns
        recent_returns = all_returns.iloc[-n:]
        
        if return_type == 'portfolio':
            # Calculate cumulative returns for the portfolio
            cumulative_returns = (1 + recent_returns).cumprod() - 1
            return cumulative_returns
        elif return_type == 'individual':
            # Calculate cumulative returns for each individual asset
            cumulative_returns = (1 + recent_returns).cumprod() - 1
            return cumulative_returns
    
    def rolling_sortino_ratio(self, n=21):
        """
        Calculates the rolling Sortino ratio of the portfolio.

        Args:
            n (int): The number of periods to consider in each rolling window.

        Returns:
            pd.Series: The rolling Sortino ratio of the portfolio.
        """
        # Calculate the daily returns of the portfolio
        returns = self.returns()
        
        # Calculate the daily downside deviation
        downside_deviation = returns.rolling(window=n).apply(lambda x: x[x < 0].std())
        
        # Calculate the rolling Sortino ratio
        rolling_sortino = np.sqrt(n) * returns.rolling(window=n).mean() / downside_deviation
        
        return rolling_sortino

    def rolling_correlation(self, benchmark, n=21):
        """
        Calculates the rolling correlation between the portfolio and a benchmark.

        Args:
            benchmark (pd.Series): The returns of the benchmark.
            n (int): The number of periods to consider in each rolling window.

        Returns:
            pd.Series: The rolling correlation between the portfolio and the benchmark.
        """
        # Calculate the daily returns of the benchmark
        benchmark_returns = benchmark
        
        # Calculate the rolling correlation
        rolling_correlation = self.returns().rolling(window=n).corr(benchmark_returns)
        
        return rolling_correlation
    
    def standard_deviation(self):
        """
        Calculates the standard deviation of the portfolio returns.

        Returns:
            float: The standard deviation of the portfolio returns.
        """
        # Calculate the standard deviation of the portfolio returns
        return self.returns().std()
    
    def benchmark_returns_minus_portfolio_returns(self, benchmark_ticker, n = 1):
        """
        Calculates the difference between the returns of the benchmark and the portfolio.

        Args:
            benchmark (pd.Series): The returns of the benchmark.

        Returns:
            pd.Series: The difference between the returns of the benchmark and the portfolio.
        """
        
        benchmark = yf.Ticker(normalize_yf_ticker(benchmark_ticker)).history(period=self.period, interval=self.interval)['Close'].pct_change(n).dropna()
        #convert benchmark index to portfolio index
        benchmark.index = self.returns(n).index
        # Calculate the difference between the benchmark and the portfolio returns
        excess_returns = benchmark - self.returns(n)
        
        return excess_returns
    
    def plot_rolling_sortino(self, plot_assets=False, n=21):
        """
        Plots the rolling Sortino ratio of the portfolio.
        If plot_assets is True, it will also plot the rolling Sortino ratio of each asset in the portfolio.

        Args:
            plot_assets (bool): Whether to plot the rolling Sortino ratio of each asset in the portfolio.
            n (int): The number of periods to consider in each rolling window.
        """

        import plotly.graph_objects as go
        import numpy as np

        # Calculate the rolling Sortino ratio of the portfolio
        portfolio_sortino = self.rolling_sortino_ratio(n=n)

        # Compute mean and standard deviation of the portfolio's Sortino ratio
        mean_val = portfolio_sortino.mean()
        std_val = portfolio_sortino.std()

        # Create the Plotly figure
        fig = go.Figure()
        
        # Add trace for portfolio Sortino ratio as a dashed line
        fig.add_trace(go.Scatter(
            x=portfolio_sortino.index,
            y=portfolio_sortino,
            mode='lines',
            name=f'{self.portfolio_name} Sortino',
            line=dict(width=4, dash='dash')
        ))

        # Define standard deviation levels and corresponding colors for shading
        std_levels = {
            1: {"color": "yellow", "opacity": 0.5},
            2: {"color": "LightCoral", "opacity": 0.5},
        }

        # Add mean and standard deviation lines using a loop to avoid redundancy
        for i in range(-3, 4):
            if i == 0:
                # Add mean line
                fig.add_hline(
                    y=mean_val,
                    line_dash="dash",
                    line_color="Green",
                    line_width=2,
                    annotation_text="Mean",
                    annotation_position="top left"
                )
            else:
                # Add lines for +/-1ÃƒÂÃ†â€™, +/-2ÃƒÂÃ†â€™, +/-3ÃƒÂÃ†â€™
                fig.add_hline(
                    y=mean_val + i * std_val,
                    line_dash="dash",
                    line_color="Red",
                    line_width=2,
                    annotation_text=f"{'+' if i > 0 else ''}{i}ÃƒÂÃ†â€™",
                    annotation_position="top left"
                )

        # Add shaded regions between standard deviation levels
        for i in std_levels:
            # Shaded region between +iÃƒÂÃ†â€™ and +(i+1)ÃƒÂÃ†â€™
            fig.add_shape(
                type="rect",
                x0=portfolio_sortino.index.min(),
                x1=portfolio_sortino.index.max(),
                y0=mean_val + i * std_val,
                y1=mean_val + (i + 1) * std_val,
                line=dict(color="Red", width=2, dash="dash"),
                fillcolor=std_levels[i]["color"],
                opacity=std_levels[i]["opacity"]
            )
            # Shaded region between - (i+1)ÃƒÂÃ†â€™ and -iÃƒÂÃ†â€™
            fig.add_shape(
                type="rect",
                x0=portfolio_sortino.index.min(),
                x1=portfolio_sortino.index.max(),
                y0=mean_val - (i + 1) * std_val,
                y1=mean_val - i * std_val,
                line=dict(color="Red", width=2, dash="dash"),
                fillcolor=std_levels[i]["color"],
                opacity=std_levels[i]["opacity"]
            )
            
        # Add annotation to the center of the plot  
        fig.add_annotation(
            x=portfolio_sortino.index[len(portfolio_sortino) // 2],
            y=mean_val,
            text=f'Portfolio Mean: {mean_val:.2f}',
            showarrow=False,
            yshift=10
        )

        # Add annotations for each shaded region
        for i in std_levels:
            fig.add_annotation(
            x=portfolio_sortino.index[len(portfolio_sortino) // 2],
            y=mean_val + (i + 0.5) * std_val,
            text=f'Portfolio +{i}ÃƒÂÃ†â€™ to +{i+1}ÃƒÂÃ†â€™',
            showarrow=False,
            yshift=10
            )
            fig.add_annotation(
            x=portfolio_sortino.index[len(portfolio_sortino) // 2],
            y=mean_val - (i + 0.5) * std_val,
            text=f'Portfolio -{i+1}ÃƒÂÃ†â€™ to -{i}ÃƒÂÃ†â€™',
            showarrow=False,
            yshift=10
            )
        
            

        # Optionally add traces for each asset's Sortino ratio as solid lines
        if plot_assets:
            for ticker in self.tickers:
                # Calculate asset returns
                asset_returns = self.data[ticker].pct_change().dropna()

                # Calculate downside deviation for rolling window
                downside_deviation = asset_returns.rolling(window=n).apply(lambda x: x[x < 0].std())

                # Calculate rolling Sortino ratio
                rolling_sortino = np.sqrt(n) * asset_returns.rolling(window=n).mean() / downside_deviation

                # Add asset Sortino ratio trace
                fig.add_trace(go.Scatter(
                    x=rolling_sortino.index,
                    y=rolling_sortino,
                    mode='lines',
                    name=f'{ticker} Sortino',
                    line=dict(width=2)
                ))

        # Add a horizontal line at zero
        fig.add_hline(
            y=0,
            line=dict(color="Red", width=2)
        )
        
        # Create a dropdown menu, the values should be 1 year, 3 years, 5 years, 10 years. Default value should be 3 years
        # Create a dropdown menu, the values should be 1 year, 3 years, 5 years, 10 years. Default value should be 3 years
        fig.update_layout(
            updatemenus=[
            dict(
            buttons=[
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[-252], portfolio_sortino.index[-1]]}],
                label="1 Year",
                method="relayout"
            ),
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[-756], portfolio_sortino.index[-1]]}],
                label="3 Years",
                method="relayout"
            ),
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[-1260], portfolio_sortino.index[-1]]}],
                label="5 Years",
                method="relayout"
            ),
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[0], portfolio_sortino.index[-1]]}],
                label="10 Years",
                method="relayout"
            )
            ],
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.25,
            yanchor="top",
            active=1  # Set default to 3 years
            ),
            ]
        )

        # Set the default view to 3 years
        fig.update_xaxes(range=[portfolio_sortino.index[-756], portfolio_sortino.index[-1]])

        
        # Update the layout of the plot for better readability and aesthetics
        fig.update_layout(
            title=f'Rolling Sortino Ratio of {self.portfolio_name}',
            xaxis_title='Date',
            yaxis_title='Rolling Sortino Ratio',
            legend_title='Ticker',
            template='plotly_dark',
            height=1000
        )

        # Display the plot
        fig.show()

In [ ]:
# Code Block 4: Define parameters
#Define parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200

selected_time_frame = time_frame_long

CLIENT_ID = require_secret("SCHWAB_CLIENT_ID")
APP_SECRET = require_secret("SCHWAB_APP_SECRET")
CALLBACK_URL = os.getenv("SCHWAB_CALLBACK_URL", "https://127.0.0.1:8182")
TOKEN_PATH = os.getenv("SCHWAB_TOKEN_PATH", "schwab_token.json")
#callback
period = '20y'
interval = '1d'
benchmark_str = 'SPY'

In [ ]:
# Code Block 5: Login to Schwab client
#Login to client (must be done manually, you will be prompted for the url)
from schwab.auth import client_from_manual_flow
from schwab.client import Client
import json
client = client_from_manual_flow(
    api_key=CLIENT_ID,
    app_secret=APP_SECRET,
    callback_url=CALLBACK_URL,
    token_path=TOKEN_PATH
)

In [ ]:
# Code Block 6: Retrieve account and market data
#retreive  account and market data

#account information: the account infromation of course
#positions_df:        the accounts current options positions
#invested_sybmols:     all of the symbols im currently invested in
#total margin:        the total amount of money under my control for each position
#benchmark_close      the historical closing prices of the benchmark data
#portfolio_close      the historical closing prices of the portfolio
#net_direction        the "net direction" of the options portfolio per position


#general account information
account_information = client.get_accounts().json()

#retrieve options options
#-------------------------------------------------------------------------------
acct_map = client.get_account_numbers().json()
acct_hash = acct_map[0]['hashValue']
resp = client.get_account(acct_hash, fields=Client.Account.Fields.POSITIONS)
acct = resp.json()
positions = acct.get("securitiesAccount", {}).get("positions", [])
#-------------------------------------------------------------------------------

#parses positions for all options positions and formats into a simple dataframe
#-------------------------------------------------------------------------------
positions_df = pd.DataFrame()
option_metadata_rows = []
option_pattern = re.compile(r'^(?P<underlying>[A-Z]{1,6})(?P<expiration>\d{6})(?P<option_type>[CP])(?P<strike>\d{8})$')
today_normalized = pd.Timestamp.today().normalize()

for position in positions:
    position_symbol = position['instrument'].get('symbol')
    position_symbol = "".join(position_symbol.split())

    if position.get('instrument', {}).get('assetType') == 'OPTION':
        match = option_pattern.match(position_symbol)
        if match:
            expiration_code = match.group('expiration')
            expiration_date = pd.to_datetime('20' + expiration_code, format='%Y%m%d', errors='coerce')
            days_to_expiration = int((expiration_date.normalize() - today_normalized).days) if pd.notna(expiration_date) else pd.NA
            strike_price = int(match.group('strike')) / 1000
            option_metadata_rows.append(
                {
                    'symbol': position_symbol,
                    'underlying': match.group('underlying'),
                    'expiration': expiration_date,
                    'days_to_expiration': days_to_expiration,
                    'option_type': match.group('option_type'),
                    'strike': strike_price,
                    'underlying_symbol': position.get('instrument', {}).get('underlyingSymbol'),
                    'put_call': position.get('instrument', {}).get('putCall'),
                    'long_quantity': position.get('longQuantity', 0.0),
                    'short_quantity': position.get('shortQuantity', 0.0),
                    'net_quantity': position.get('longQuantity', 0.0) - position.get('shortQuantity', 0.0),
                    'average_price': position.get('averagePrice', 0.0),
                }
            )

    positions_df = pd.DataFrame(option_metadata_rows)



#calculates informally the directionality of my portfolio
#---------------------------------------------------------------------------------------------------------
if not positions_df.empty:
    direction_factor = positions_df['option_type'].map({'C': 1, 'P': -1}).fillna(0)
    positions_df['directional_value'] = positions_df['net_quantity'] * positions_df['average_price'] * direction_factor * 100
    net_direction = positions_df.groupby('underlying')['directional_value'].sum()
    sentiment_map = net_direction.apply(lambda x: 'bullish' if x > 0 else ('bearish' if x < 0 else 'neutral'))
    option_sentiment = sentiment_map.to_dict()
    net_direction = net_direction.to_frame('directional_value').join(sentiment_map.rename('sentiment'))
    #display(net_direction.to_frame('directional_value').join(sentiment_map.rename('sentiment')))
else:
    option_sentiment = {}
    net_direction = pd.DataFrame(columns=['directional_value', 'sentiment'])
    
net_direction['sign'] = net_direction['sentiment'].map({'bullish': 1, 'bearish': -1, 'neutral': 0})
#--------------------------------------------------------------------------------------------------------



#retrieve invested symbols
#---------------------------------------------------------------------------------
organized_positions = {}
for pos in positions:
    symbol = pos.get("instrument", {}).get("underlyingSymbol", "")
    if symbol:
        organized_positions.setdefault(symbol, []).append(pos)
        
invested_symbols = list(organized_positions.keys())
#---------------------------------------------------------------------------------


#for each symbol in organized_positions, get the total invested amount
net_invested_amounts = {}
total_margin = {}

for symbol in invested_symbols:
    positions_list = organized_positions[symbol]
    net_sum = 0
    #print(positions_list[0].keys())
    
    for position in positions_list:
        averagePrice = position.get("averagePrice", 0)
        maintenance_requirement = position.get("maintenanceRequirement", 0) / 100
        putCall = position.get("instrument", {}).get("putCall", "N/A")
        short_quantity = position.get("shortQuantity", 0)
        long_quantity = position.get("longQuantity", 0)
        # If it's a PUT, treat price as negative
        if putCall == "PUT":
            averagePrice = -averagePrice

        # Calculate net invested amount using both long and short quantities
        net_invested = (long_quantity - short_quantity) * averagePrice * 100  # times 100 for options
        maintenance_requirement = maintenance_requirement * 100  # times 100 for options
        total_margin[symbol] = total_margin.get(symbol, 0) + maintenance_requirement

        # Print symbol, putCall, averagePrice, short_quantity, long_quantity, and net_invested
        #print(f"Symbol: {symbol}, Put/Call: {putCall}, Average Price: {averagePrice}, Short Quantity: {short_quantity}, Long Quantity: {long_quantity}, Net Invested: {net_invested}")

        net_sum += net_invested
    net_invested_amounts[symbol] = net_sum
    #display(f"Symbol: {symbol}, Net Invested Amount: {net_sum}")
    #display(f"Symbol: {symbol}, Total Margin Requirement: {total_margin[symbol]}")
#-------------------------------------------------------------------------------------

#retrieve core data for benchmark and portfolio
#--------------------------------------------------------------------------
benchmark_data = yf.Ticker('SPY').history(period=period, interval=interval)
invested_symbol_map = build_yf_ticker_map(invested_symbols)
portfolio_data = yf.download(
            tickers=list(invested_symbol_map.values()),
            period=period,
            interval=interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )

portfolio_closing_prices = portfolio_data['Close']
if isinstance(portfolio_closing_prices, pd.Series):
    portfolio_closing_prices = portfolio_closing_prices.to_frame(name=invested_symbols[0])
else:
    portfolio_closing_prices = portfolio_closing_prices.rename(columns={yf_ticker: ticker for ticker, yf_ticker in invested_symbol_map.items()})
benchmark_close          = benchmark_data['Close']

#portfolioc_closing prices index is formated like 'YYYY-MM-DD' but benchmark_close index is formated like 'YYYY-MM-DD HH:MM:SS-00:00' 
#need to convert portfolio_closing_prices index to match benchmark_close index

#display(portfolio_closing_prices.index)
portfolio_closing_prices.index = portfolio_closing_prices.index.tz_localize(None)
benchmark_close.index = benchmark_close.index.tz_localize(None)
#--------------------------------------------------------------------------
net_direction
raw_prices = portfolio_closing_prices.copy()


In [ ]:
# Retrieve net cost basis for each option grouped by ticker

# net cost basis = net_quantity * average_price * 100 (per contract)



if not positions_df.empty:

    positions_df['net_cost_basis'] = positions_df['net_quantity'] * positions_df['average_price'] * 100

    net_cost_basis = positions_df.groupby('underlying')['net_cost_basis'].sum().rename('net_cost_basis')

    display(net_cost_basis.to_frame())

else:

    net_cost_basis = pd.Series(dtype=float, name='net_cost_basis')

    print("No options positions found.")



available_option_underlyings = (

    sorted(positions_df['underlying'].dropna().unique().tolist())

    if not positions_df.empty else []

)

benchmark_label_for_beta = benchmark_str if 'benchmark_str' in globals() else 'SPY'

portfolio_mode_options = [

    ('ALL - Equal Move', 'ALL'),

    (f'ALL - Beta Weighted vs {benchmark_label_for_beta} %', 'ALL_BETA'),

    (f'ALL - Beta Weighted vs {benchmark_label_for_beta} (Price)', 'ALL_BETA_PRICE'),

]

ticker_options = (

    portfolio_mode_options + [(underlying, underlying) for underlying in available_option_underlyings]

    if available_option_underlyings else []

)

strike_range_options = [

    ('All', 'All'),

    ('+-1', 1),

    ('+-2', 2),

    ('+-5', 5),

    ('+-10', 10),

    ('+-25', 25),

    ('+-50', 50),

    ('+-100', 100),

]

default_underlying = 'URA' if 'URA' in available_option_underlyings else ('ALL' if available_option_underlyings else None)



portfolio_latest_prices = (

    portfolio_closing_prices.ffill().iloc[-1].dropna().to_dict()

    if isinstance(portfolio_closing_prices, pd.DataFrame) and not portfolio_closing_prices.empty

    else {}

)

benchmark_current_price = (

    float(benchmark_close.ffill().iloc[-1])

    if isinstance(benchmark_close, pd.Series) and not benchmark_close.dropna().empty

    else np.nan

)

benchmark_returns_for_beta = (

    benchmark_close.pct_change().dropna()

    if isinstance(benchmark_close, pd.Series)

    else pd.Series(dtype=float)

)

asset_returns_for_beta = (

    portfolio_closing_prices.pct_change()

    if isinstance(portfolio_closing_prices, pd.DataFrame) and not portfolio_closing_prices.empty

    else pd.DataFrame()

)

underlying_beta_map = {}

if not asset_returns_for_beta.empty and not benchmark_returns_for_beta.empty:

    for underlying_name in asset_returns_for_beta.columns:

        aligned_returns = pd.concat(

            [

                asset_returns_for_beta[underlying_name].rename('asset'),

                benchmark_returns_for_beta.rename('benchmark'),

            ],

            axis=1,

        ).dropna()



        if len(aligned_returns) < 2 or aligned_returns['benchmark'].var() == 0:

            underlying_beta_map[underlying_name] = 1.0

            continue



        beta_value = aligned_returns['asset'].cov(aligned_returns['benchmark']) / aligned_returns['benchmark'].var()

        if pd.notna(beta_value) and np.isfinite(beta_value):

            underlying_beta_map[underlying_name] = float(beta_value)

        else:

            underlying_beta_map[underlying_name] = 1.0

else:

    underlying_beta_map = {underlying_name: 1.0 for underlying_name in available_option_underlyings}





def get_reference_price(underlying_name, legs_frame):

    current_price = portfolio_latest_prices.get(underlying_name, np.nan)

    if pd.notna(current_price):

        return float(current_price)



    strikes = legs_frame['strike'].dropna()

    if not strikes.empty:

        return float(strikes.mean())



    return np.nan





def get_underlying_beta(underlying_name):

    beta_value = underlying_beta_map.get(underlying_name, 1.0)

    if pd.isna(beta_value) or not np.isfinite(beta_value):

        return 1.0

    return float(beta_value)





def select_strikes_for_frame(legs_frame, reference_price, selected_strike_range):

    unique_strikes = np.sort(legs_frame['strike'].dropna().unique())

    if len(unique_strikes) == 0:

        return unique_strikes, np.nan, np.nan, 'No strikes'



    if selected_strike_range == 'All':

        return unique_strikes, float(unique_strikes.min()), float(unique_strikes.max()), 'All strikes'



    strike_range = float(selected_strike_range)

    lower_bound = reference_price - strike_range

    upper_bound = reference_price + strike_range

    selected_strikes = unique_strikes[(unique_strikes >= lower_bound) & (unique_strikes <= upper_bound)]



    if len(selected_strikes) == 0:

        nearest_idx = np.argmin(np.abs(unique_strikes - reference_price))

        selected_strikes = np.array([unique_strikes[nearest_idx]])

        lower_bound = float(selected_strikes[0])

        upper_bound = float(selected_strikes[0])



    return selected_strikes, float(lower_bound), float(upper_bound), f'+/- {strike_range:g}'





def plot_option_expiration_pl(selected_underlying, selected_strike_range):

    global fig_option_pl, option_legs, current_px, target_underlying



    target_underlying = selected_underlying

    current_px = np.nan

    use_portfolio_mode = selected_underlying in {'ALL', 'ALL_BETA', 'ALL_BETA_PRICE'}

    use_beta_weighting = selected_underlying in {'ALL_BETA', 'ALL_BETA_PRICE'}

    use_beta_price_axis = selected_underlying == 'ALL_BETA_PRICE'



    if use_portfolio_mode:

        scoped_legs = positions_df.copy()

    else:

        scoped_legs = positions_df[positions_df['underlying'] == selected_underlying].copy()



    if scoped_legs.empty:

        if use_portfolio_mode:

            print('No option positions found in positions_df.')

        else:

            print(f"No {selected_underlying} option positions found in positions_df.")

        return



    fig_option_pl = go.Figure()



    if use_portfolio_mode:

        reference_price_map = {}

        selected_frames = []

        total_unique_strikes = 0

        selected_strike_count = 0



        for underlying_name, group in scoped_legs.groupby('underlying'):

            reference_price = get_reference_price(underlying_name, group)

            if pd.isna(reference_price):

                continue



            reference_price_map[underlying_name] = reference_price

            selected_strikes, _, _, _ = select_strikes_for_frame(group, reference_price, selected_strike_range)

            total_unique_strikes += group['strike'].dropna().nunique()

            selected_strike_count += len(selected_strikes)

            selected_frames.append(group[group['strike'].isin(selected_strikes)].copy())



        if not selected_frames:

            print('No portfolio option legs matched the selected strike filter.')

            return



        option_legs = pd.concat(selected_frames, ignore_index=True)



        scenario_trigger_levels = []

        for underlying_name, group in option_legs.groupby('underlying'):

            reference_price = reference_price_map.get(underlying_name, np.nan)

            if not pd.notna(reference_price) or reference_price <= 0:

                continue



            relative_moves = group['strike'].dropna() / reference_price - 1

            if use_beta_weighting:

                beta_value = get_underlying_beta(underlying_name)

                if abs(beta_value) > 1e-6:

                    scenario_trigger_levels.extend((relative_moves / beta_value).tolist())

            else:

                scenario_trigger_levels.extend(relative_moves.tolist())



        if scenario_trigger_levels:

            scenario_min = min(min(scenario_trigger_levels), -0.05)

            scenario_max = max(max(scenario_trigger_levels), 0.05)

        else:

            scenario_min, scenario_max = -0.25, 0.25



        pad = max((scenario_max - scenario_min) * 0.25, 0.05)

        scenario_move_range = np.linspace(

            max(scenario_min - pad, -0.95),

            min(scenario_max + pad, 0.95),

            500,

        )

        benchmark_price_range = np.maximum(benchmark_current_price * (1 + scenario_move_range), 0)

        if use_beta_price_axis:

            current_px = benchmark_current_price

            scenario_x_values = benchmark_price_range

        else:

            scenario_x_values = scenario_move_range * 100

        total_pl = np.zeros(len(scenario_move_range))



        for _, leg in option_legs.iterrows():

            underlying_name = leg['underlying']

            reference_price = reference_price_map.get(underlying_name, np.nan)

            if pd.isna(reference_price):

                continue



            strike = leg['strike']

            opt_type = leg['option_type']

            qty = leg['net_quantity']

            avg_px = leg['average_price']

            exp_label = leg['expiration'].strftime('%b %d') if pd.notna(leg['expiration']) else 'No Exp'

            beta_value = get_underlying_beta(underlying_name)



            if use_beta_weighting:

                scenario_prices = np.maximum(reference_price * (1 + beta_value * scenario_move_range), 0)

                leg_label_prefix = f"{underlying_name} (beta {beta_value:.2f})"

            else:

                scenario_prices = np.maximum(reference_price * (1 + scenario_move_range), 0)

                leg_label_prefix = underlying_name



            if opt_type == 'C':

                intrinsic = np.maximum(scenario_prices - strike, 0)

            else:

                intrinsic = np.maximum(strike - scenario_prices, 0)



            leg_pl = qty * (intrinsic - avg_px) * 100

            total_pl += leg_pl



            direction = 'Long' if qty > 0 else 'Short'

            fig_option_pl.add_trace(go.Scatter(

                x=scenario_x_values,

                y=leg_pl,

                mode='lines',

                name=f"{leg_label_prefix} | {direction} {abs(qty):.0f}x {strike:.0f}{opt_type} exp {exp_label}",

                line=dict(width=1, dash='dot'),

                opacity=0.6,

                visible='legendonly',

            ))



        if use_beta_price_axis:

            portfolio_trace_name = f'Total Portfolio P/L (Beta Weighted vs {benchmark_label_for_beta} Price)'

        elif use_beta_weighting:

            portfolio_trace_name = f'Total Portfolio P/L (Beta Weighted vs {benchmark_label_for_beta} %)'

        else:

            portfolio_trace_name = 'Total Portfolio P/L'

        fig_option_pl.add_trace(go.Scatter(

            x=scenario_x_values,

            y=total_pl,

            mode='lines',

            name=portfolio_trace_name,

            line=dict(width=3, color='white'),

        ))

        fig_option_pl.add_hline(y=0, line_width=1, line_dash='dash', line_color='gray')

        current_marker_x = benchmark_current_price if use_beta_price_axis else 0

        if use_beta_price_axis:

            current_marker_text = f'Current {benchmark_label_for_beta}: {benchmark_current_price:.2f}'

        elif use_beta_weighting:

            current_marker_text = f'Current {benchmark_label_for_beta}: 0%'

        else:

            current_marker_text = 'Current prices'

        fig_option_pl.add_vline(

            x=current_marker_x,

            line_width=1,

            line_dash='dash',

            line_color='yellow',

            annotation_text=current_marker_text,

            annotation_position='top right',

        )



        displayed_cost = option_legs['net_cost_basis'].sum() if 'net_cost_basis' in option_legs.columns else np.nan

        total_cost = float(net_cost_basis.sum()) if not net_cost_basis.empty else np.nan

        range_text = 'All strikes' if selected_strike_range == 'All' else f"+/- {float(selected_strike_range):g} around each underlying current price"

        if use_beta_price_axis:

            mode_text = f"Beta weighted to {benchmark_label_for_beta} with {benchmark_label_for_beta} price axis"

        elif use_beta_weighting:

            mode_text = f"Beta weighted to {benchmark_label_for_beta} with {benchmark_label_for_beta} move % axis"

        else:

            mode_text = 'Equal % move across all underlyings'

        subtitle = (

            f"Mode: {mode_text}"

            f" | Range: {range_text}"

            f" | Underlyings shown: {option_legs['underlying'].nunique()} of {scoped_legs['underlying'].nunique()}"

            f" | Strike groups shown: {selected_strike_count} of {total_unique_strikes}"

            f" | Displayed cost basis: ${displayed_cost:,.2f}"

            f" | Total portfolio cost basis: ${total_cost:,.2f}"

        )

        if use_beta_price_axis:

            title_text = f'Entire Portfolio Options — Beta-Weighted P/L vs {benchmark_label_for_beta} Price at Expiration'

            xaxis_title = f'{benchmark_label_for_beta} Price at Expiration'

        elif use_beta_weighting:

            title_text = f'Entire Portfolio Options — Beta-Weighted P/L vs {benchmark_label_for_beta} Move % at Expiration'

            xaxis_title = f'{benchmark_label_for_beta} Move at Expiration (%)'

        else:

            title_text = 'Entire Portfolio Options — Equal-Move P/L at Expiration'

            xaxis_title = 'Underlying Move at Expiration (%)'



        fig_option_pl.update_layout(

            title=f'{title_text}<br><sup>{subtitle}</sup>',

            xaxis_title=xaxis_title,

            yaxis_title='P/L ($)',

            template='plotly_dark',

            hovermode='x unified',

            height=700,

            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),

        )

        fig_option_pl.show()

        return



    reference_price = get_reference_price(selected_underlying, scoped_legs)

    current_px = portfolio_latest_prices.get(selected_underlying, np.nan)

    selected_strikes, lower_bound, upper_bound, range_label = select_strikes_for_frame(

        scoped_legs,

        reference_price,

        selected_strike_range,

    )

    total_unique_strikes = scoped_legs['strike'].dropna().nunique()

    option_legs = scoped_legs[scoped_legs['strike'].isin(selected_strikes)].copy()



    if option_legs.empty:

        print(f"No {selected_underlying} option legs matched the selected strike filter.")

        return



    strike_min = option_legs['strike'].min()

    strike_max = option_legs['strike'].max()

    reference_min = min(strike_min, current_px) if pd.notna(current_px) else strike_min

    reference_max = max(strike_max, current_px) if pd.notna(current_px) else strike_max

    pad = max((reference_max - reference_min) * 0.25, reference_max * 0.10, 1.0)

    s_range = np.linspace(max(reference_min - pad, 0), reference_max + pad, 500)

    total_pl = np.zeros(len(s_range))



    for _, leg in option_legs.iterrows():

        strike = leg['strike']

        opt_type = leg['option_type']

        qty = leg['net_quantity']

        avg_px = leg['average_price']

        exp_label = leg['expiration'].strftime('%b %d') if pd.notna(leg['expiration']) else 'No Exp'



        if opt_type == 'C':

            intrinsic = np.maximum(s_range - strike, 0)

        else:

            intrinsic = np.maximum(strike - s_range, 0)



        leg_pl = qty * (intrinsic - avg_px) * 100

        total_pl += leg_pl



        direction = 'Long' if qty > 0 else 'Short'

        fig_option_pl.add_trace(go.Scatter(

            x=s_range,

            y=leg_pl,

            mode='lines',

            name=f"{direction} {abs(qty):.0f}x {strike:.0f}{opt_type} exp {exp_label}",

            line=dict(width=1, dash='dot'),

            opacity=0.6,

            visible='legendonly',

        ))



    fig_option_pl.add_trace(go.Scatter(

        x=s_range,

        y=total_pl,

        mode='lines',

        name='Total P/L',

        line=dict(width=3, color='white'),

    ))

    fig_option_pl.add_hline(y=0, line_width=1, line_dash='dash', line_color='gray')



    if pd.notna(current_px):

        fig_option_pl.add_vline(

            x=current_px,

            line_width=1,

            line_dash='dash',

            line_color='yellow',

            annotation_text=f'Current: {current_px:.2f}',

            annotation_position='top right',

        )



    total_cost = net_cost_basis.get(selected_underlying, np.nan)

    displayed_cost = option_legs['net_cost_basis'].sum() if 'net_cost_basis' in option_legs.columns else np.nan

    subtitle = (

        f"Range: {range_label} around {reference_price:.2f}"

        f" | Window: [{lower_bound:.2f}, {upper_bound:.2f}]"

        f" | Strikes shown: {len(selected_strikes)} of {total_unique_strikes}"

        f" | Displayed cost basis: ${displayed_cost:,.2f}"

        f" | Total ticker cost basis: ${total_cost:,.2f}"

    )



    fig_option_pl.update_layout(

        title=f'{selected_underlying} Options — P/L at Expiration<br><sup>{subtitle}</sup>',

        xaxis_title=f'{selected_underlying} Price at Expiration',

        yaxis_title='P/L ($)',

        template='plotly_dark',

        hovermode='x unified',

        height=500,

        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),

    )

    fig_option_pl.show()





if not available_option_underlyings:

    print('No option positions found in positions_df.')

else:

    try:

        import ipywidgets as widgets



        ticker_dropdown = widgets.Dropdown(

            options=ticker_options,

            value=default_underlying,

            description='Ticker:',

            layout=widgets.Layout(width='380px'),

        )

        strike_range_dropdown = widgets.Dropdown(

            options=strike_range_options,

            value='All',

            description='Range:',

            layout=widgets.Layout(width='220px'),

        )



        controls = widgets.HBox([ticker_dropdown, strike_range_dropdown])

        output = widgets.interactive_output(

            plot_option_expiration_pl,

            {

                'selected_underlying': ticker_dropdown,

                'selected_strike_range': strike_range_dropdown,

            },

        )

        display(widgets.VBox([controls, output]))

    except ImportError:

        print('ipywidgets is not available in this notebook kernel. Showing the default selection with all strikes instead.')

        plot_option_expiration_pl(default_underlying, 'All')


In [ ]:
# Code Block 7: Option leg coverage audit
# =========================
# 7) Verify every raw Schwab option leg is represented in positions_df
# =========================

raw_option_rows = []
regex_miss_rows = []

for position in positions:
    instrument = position.get('instrument', {})
    if instrument.get('assetType') != 'OPTION':
        continue

    raw_symbol = instrument.get('symbol', '')
    normalized_symbol = ''.join((raw_symbol or '').split())
    match = option_pattern.match(normalized_symbol)

    expiration = pd.NaT
    parsed_option_type = pd.NA
    parsed_strike = pd.NA
    if match:
        expiration = pd.to_datetime('20' + match.group('expiration'), format='%Y%m%d', errors='coerce')
        parsed_option_type = match.group('option_type')
        parsed_strike = int(match.group('strike')) / 1000

    row = {
        'raw_symbol': raw_symbol,
        'normalized_symbol': normalized_symbol,
        'underlying_symbol': instrument.get('underlyingSymbol'),
        'put_call': instrument.get('putCall'),
        'expiration': expiration,
        'parsed_option_type': parsed_option_type,
        'parsed_strike': parsed_strike,
        'long_quantity': position.get('longQuantity', 0.0),
        'short_quantity': position.get('shortQuantity', 0.0),
        'average_price': position.get('averagePrice', 0.0),
    }
    raw_option_rows.append(row)

    if not match:
        regex_miss_rows.append(dict(row, drop_reason='symbol_did_not_match_option_pattern'))

raw_option_df = pd.DataFrame(raw_option_rows)
parsed_option_df = positions_df.copy()

if raw_option_df.empty:
    print('[Code Block 7] No raw option legs returned by Schwab for the selected account.')
else:
    raw_audit = (
        raw_option_df.groupby('normalized_symbol', dropna=False, as_index=False)
        .agg(
            raw_rows=('normalized_symbol', 'size'),
            raw_long_quantity=('long_quantity', 'sum'),
            raw_short_quantity=('short_quantity', 'sum'),
            raw_average_price=('average_price', 'first'),
            underlying_symbol=('underlying_symbol', 'first'),
            expiration=('expiration', 'first'),
            put_call=('put_call', 'first'),
        )
    )

    if parsed_option_df.empty:
        parsed_audit = pd.DataFrame(columns=[
            'symbol',
            'parsed_rows',
            'parsed_long_quantity',
            'parsed_short_quantity',
            'parsed_average_price',
        ])
    else:
        parsed_option_df['symbol'] = parsed_option_df['symbol'].astype(str)
        parsed_audit = (
            parsed_option_df.groupby('symbol', as_index=False)
            .agg(
                parsed_rows=('symbol', 'size'),
                parsed_long_quantity=('long_quantity', 'sum'),
                parsed_short_quantity=('short_quantity', 'sum'),
                parsed_average_price=('average_price', 'first'),
            )
        )

    audit = raw_audit.merge(parsed_audit, left_on='normalized_symbol', right_on='symbol', how='left')
    audit[['parsed_rows', 'parsed_long_quantity', 'parsed_short_quantity']] = audit[
        ['parsed_rows', 'parsed_long_quantity', 'parsed_short_quantity']
    ].fillna(0)
    audit['raw_rows'] = audit['raw_rows'].fillna(0)
    audit['row_match'] = audit['raw_rows'].astype(int) == audit['parsed_rows'].astype(int)
    audit['long_match'] = audit['raw_long_quantity'].round(6) == audit['parsed_long_quantity'].round(6)
    audit['short_match'] = audit['raw_short_quantity'].round(6) == audit['parsed_short_quantity'].round(6)
    audit['present_in_positions_df'] = audit[['row_match', 'long_match', 'short_match']].all(axis=1)

    raw_symbol_count = raw_option_df['normalized_symbol'].nunique()
    parsed_symbol_count = parsed_option_df['symbol'].nunique() if not parsed_option_df.empty else 0

    print('[Code Block 7] Option Leg Coverage Audit')
    print(f'- Accounts returned by Schwab: {len(acct_map):,}')
    print(f'- Selected account hash: {acct_hash}')
    print(f'- Raw option legs from Schwab: {len(raw_option_df):,}')
    print(f'- Parsed option legs in positions_df: {len(parsed_option_df):,}')
    print(f'- Distinct option symbols from Schwab: {raw_symbol_count:,}')
    print(f'- Distinct option symbols in positions_df: {parsed_symbol_count:,}')
    print(f'- Regex misses: {len(regex_miss_rows):,}')

    if audit['present_in_positions_df'].all():
        print('Status: PASS - every raw Schwab option leg is represented in positions_df.')
    else:
        print('Status: FAIL - some option legs are missing or quantities do not match.')
        display(
            audit.loc[
                ~audit['present_in_positions_df'],
                [
                    'underlying_symbol',
                    'normalized_symbol',
                    'expiration',
                    'put_call',
                    'raw_rows',
                    'parsed_rows',
                    'raw_long_quantity',
                    'parsed_long_quantity',
                    'raw_short_quantity',
                    'parsed_short_quantity',
                ],
            ].sort_values(['underlying_symbol', 'expiration', 'normalized_symbol'])
        )

    if regex_miss_rows:
        print('Fix suggestion: some option symbols failed the OCC regex, so keep those legs using instrument fields even when regex parsing fails.')
        display(pd.DataFrame(regex_miss_rows).sort_values(['underlying_symbol', 'raw_symbol']))

    if len(acct_map) > 1:
        print('Fix suggestion: your code currently pulls acct_map[0]. If your broker UI is showing another account, choose the matching hash or loop through every account.')


In [ ]:
# Code Block 8: DTE ladder
# =========================
# 8) Options expiration ladder
# =========================

if positions_df.empty:
    print('No option positions available for the DTE ladder.')
else:
    dte_view = positions_df.copy()
    dte_view['expiration'] = pd.to_datetime(dte_view['expiration'], errors='coerce')
    dte_view['days_to_expiration'] = pd.to_numeric(dte_view['days_to_expiration'], errors='coerce')
    dte_view = dte_view.dropna(subset=['underlying', 'expiration', 'days_to_expiration', 'net_quantity'])

    if dte_view.empty:
        print('No valid option rows available for the DTE ladder.')
    else:
        dte_view['days_to_expiration'] = dte_view['days_to_expiration'].astype(int)
        dte_view['gross_contracts'] = dte_view['net_quantity'].abs()
        dte_view['expiration_label'] = (
            dte_view['expiration'].dt.strftime('%Y-%m-%d')
            + ' | '
            + dte_view['days_to_expiration'].astype(str)
            + ' DTE'
        )

        expiration_order_frame = (
            dte_view[['expiration_label', 'expiration', 'days_to_expiration']]
            .drop_duplicates()
            .sort_values(['expiration', 'days_to_expiration'])
        )
        expiration_order = expiration_order_frame['expiration_label'].tolist()
        underlying_order = (
            dte_view.groupby('underlying')['gross_contracts']
            .sum()
            .sort_values(ascending=False)
            .index
            .tolist()
        )

        ladder_frame = (
            dte_view.groupby(['expiration_label', 'underlying'], as_index=False)
            .agg(
                gross_contracts=('gross_contracts', 'sum'),
                days_to_expiration=('days_to_expiration', 'first'),
                expiration=('expiration', 'first'),
            )
        )
        ladder_frame['expiration_label'] = pd.Categorical(
            ladder_frame['expiration_label'],
            categories=expiration_order,
            ordered=True,
        )
        ladder_frame = ladder_frame.sort_values(['expiration_label', 'underlying'])

        full_dte_range = list(range(0, int(dte_view['days_to_expiration'].max()) + 1))
        dense_dtick = 1 if len(full_dte_range) <= 31 else 5 if len(full_dte_range) <= 120 else 10 if len(full_dte_range) <= 260 else 25
        dte_axis_frame = (
            dte_view.groupby(['days_to_expiration', 'underlying'], as_index=False)['gross_contracts']
            .sum()
        )
        dte_expiration_lookup = (
            dte_view[['days_to_expiration', 'expiration']]
            .drop_duplicates()
            .sort_values(['days_to_expiration', 'expiration'])
            .assign(expiration=lambda frame: frame['expiration'].dt.strftime('%Y-%m-%d'))
            .groupby('days_to_expiration')['expiration']
            .agg(', '.join)
            .reindex(full_dte_range, fill_value='No listed expiration')
        )
        dense_expiration_labels = dte_expiration_lookup.tolist()
        sparse_hovertemplate = 'Underlying: %{fullData.name}<br>Expiration: %{x}<br>Gross Contracts: %{y:.2f}<extra></extra>'
        dense_hovertemplate = 'Underlying: %{fullData.name}<br>DTE: %{x}<br>Expiration(s): %{customdata}<br>Gross Contracts: %{y:.2f}<extra></extra>'

        print('[Code Block 8] Expiration Ladder Summary:')
        print(f"- Option rows: {len(dte_view):,}")
        print(f"- Unique underlyings: {dte_view['underlying'].nunique():,}")
        print(f"- Expiration dates tracked: {dte_view['expiration'].nunique():,}")
        print(f"- Nearest expiration: {dte_view['days_to_expiration'].min()} DTE")
        print(f"- Furthest expiration: {dte_view['days_to_expiration'].max()} DTE")
        print(f"- Gross contracts tracked: {dte_view['gross_contracts'].sum():,.2f}")

        sparse_x_data = []
        sparse_y_data = []
        sparse_customdata = []
        sparse_hovertemplates = []
        dense_x_data = []
        dense_y_data = []
        dense_customdata = []
        dense_hovertemplates = []

        fig = go.Figure()
        for underlying in underlying_order:
            sparse_slice = ladder_frame[ladder_frame['underlying'] == underlying].sort_values('days_to_expiration')
            if sparse_slice.empty:
                continue

            dense_slice = (
                dte_axis_frame[dte_axis_frame['underlying'] == underlying]
                .set_index('days_to_expiration')
                .reindex(full_dte_range, fill_value=0.0)
                .rename_axis('days_to_expiration')
                .reset_index()
            )

            sparse_x = sparse_slice['expiration_label'].astype(str).tolist()
            sparse_y = sparse_slice['gross_contracts'].tolist()
            dense_x = dense_slice['days_to_expiration'].tolist()
            dense_y = dense_slice['gross_contracts'].tolist()

            sparse_x_data.append(sparse_x)
            sparse_y_data.append(sparse_y)
            sparse_customdata.append([None] * len(sparse_x))
            sparse_hovertemplates.append(sparse_hovertemplate)
            dense_x_data.append(dense_x)
            dense_y_data.append(dense_y)
            dense_customdata.append(dense_expiration_labels)
            dense_hovertemplates.append(dense_hovertemplate)

            fig.add_trace(
                go.Bar(
                    x=sparse_x,
                    y=sparse_y,
                    name=underlying,
                    customdata=[None] * len(sparse_x),
                    hovertemplate=sparse_hovertemplate,
                )
            )

        dte_marker_lines = [21, 50, 200]
        dense_shapes = [
            dict(
                type='line',
                x0=dte_marker,
                x1=dte_marker,
                y0=0,
                y1=1,
                xref='x',
                yref='paper',
                line=dict(color=color, width=2, dash='dash'),
            )
            for dte_marker, color in zip(dte_marker_lines, ['#00CC96', '#FECB52', '#EF553B'])
            if dte_marker <= full_dte_range[-1]
        ]

        fig.update_layout(
            title='Options Expiration Ladder',
            template='plotly_dark',
            height=760,
            barmode='stack',
            legend=dict(
                orientation='h',
                yanchor='top',
                y=-0.2,
                xanchor='left',
                x=0,
                font=dict(size=9),
            ),
            margin=dict(l=50, r=50, t=135, b=120),
            updatemenus=[
                dict(
                    buttons=[
                        dict(
                            label='By Expiration',
                            method='update',
                            args=[
                                {
                                    'x': sparse_x_data,
                                    'y': sparse_y_data,
                                    'customdata': sparse_customdata,
                                    'hovertemplate': sparse_hovertemplates,
                                },
                                {
                                    'title': {'text': 'Options Expiration Ladder'},
                                    'shapes': [],
                                    'xaxis': dict(
                                        title='Expiration | DTE',
                                        tickangle=-35,
                                        type='category',
                                        categoryorder='array',
                                        categoryarray=expiration_order,
                                    ),
                                },
                            ],
                        ),
                        dict(
                            label='Full DTE Axis',
                            method='update',
                            args=[
                                {
                                    'x': dense_x_data,
                                    'y': dense_y_data,
                                    'customdata': dense_customdata,
                                    'hovertemplate': dense_hovertemplates,
                                },
                                {
                                    'title': {'text': 'Options Expiration Ladder | Full DTE Axis'},
                                    'shapes': dense_shapes,
                                    'xaxis': dict(
                                        title='Days to Expiration',
                                        tickangle=0,
                                        type='linear',
                                        range=[-0.5, full_dte_range[-1] + 0.5],
                                        dtick=dense_dtick,
                                    ),
                                },
                            ],
                        ),
                    ],
                    direction='down',
                    showactive=True,
                    x=0.5,
                    xanchor='center',
                    y=1.14,
                    yanchor='top',
                )
            ],
        )
        fig.update_xaxes(
            title_text='Expiration | DTE',
            tickangle=-35,
            type='category',
            categoryorder='array',
            categoryarray=expiration_order,
        )
        fig.update_yaxes(title_text='Gross Contracts')
        fig.show()


In [ ]:
# Code Block 9: Asset-level signed and unsigned analytics
# =========================
# 9) Asset-level signed and unsigned analytics
# =========================

rolling_windows = (21, 50, 200)
benchmark_label = 'SPY'

def _rolling_sharpe(values, window):
    rolling_mean = values.rolling(window).mean()
    rolling_std = values.rolling(window).std(ddof=0)
    sharpe = rolling_mean.div(rolling_std).mul(np.sqrt(252))
    return sharpe.mask(rolling_std.eq(0), 0.0).replace([np.inf, -np.inf], np.nan)

def _apply_zscore(values):
    if isinstance(values, pd.Series):
        return z_score(values)
    return values.apply(z_score)

def _latest_sorted_snapshot(frame):
    valid = frame.dropna(how='all')
    if valid.empty:
        return pd.Series(dtype=float)
    return valid.iloc[-1].sort_values(ascending=False)

def _log_summary(label, values):
    if isinstance(values, pd.DataFrame):
        valid = values.dropna(how='all')
        if valid.empty:
            return f'- {label}: empty DataFrame'
        return (
            f'- {label}: DataFrame {valid.shape[0]:,} x {valid.shape[1]} '
            f'({valid.index.min():%Y-%m-%d} to {valid.index.max():%Y-%m-%d})'
        )
    valid = values.dropna()
    if valid.empty:
        return f'- {label}: empty Series'
    return f'- {label}: Series {valid.shape[0]:,} rows ({valid.index.min():%Y-%m-%d} to {valid.index.max():%Y-%m-%d})'

# Daily return streams: unsigned raw returns and signed tradable returns
daily_returns = portfolio_closing_prices.pct_change().dropna(how='all')
benchmark_daily_returns = benchmark_close.pct_change().dropna()
signs = (
    net_direction['sign']
    .reindex(portfolio_closing_prices.columns)
    .fillna(1.0)
    .astype(float)
)
daily_returns_signed = daily_returns.mul(signs, axis=1)
benchmark_daily_returns_aligned = benchmark_daily_returns.reindex(daily_returns.index).dropna()
daily_returns_for_corr = daily_returns.reindex(benchmark_daily_returns_aligned.index)
daily_returns_signed_for_corr = daily_returns_signed.reindex(benchmark_daily_returns_aligned.index)
portfolio_daily_returns_for_corr = daily_returns_signed.mean(axis=1).reindex(benchmark_daily_returns_aligned.index)
print(f'[Code Block 9] Calculated daily return streams for {len(portfolio_closing_prices.columns)} assets.')

# Assets / Rolling horizon returns (unsigned + signed)
asset_return_windows = {
    window: portfolio_closing_prices.pct_change(window).dropna(how='all')
    for window in rolling_windows
}
asset_signed_return_windows = {
    window: frame.mul(signs, axis=1)
    for window, frame in asset_return_windows.items()
}
asset_return_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_return_windows.items()
}
asset_signed_return_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_return_windows.items()
}

portfolio_assets_rolling_returns_21 = asset_return_windows[21]
portfolio_assets_rolling_returns_50 = asset_return_windows[50]
portfolio_assets_rolling_returns_200 = asset_return_windows[200]
portfolio_assets_rolling_signed_returns_21 = asset_signed_return_windows[21]
portfolio_assets_rolling_signed_returns_50 = asset_signed_return_windows[50]
portfolio_assets_rolling_signed_returns_200 = asset_signed_return_windows[200]
portfolio_assets_rolling_returns_z_scores_21 = asset_return_z_windows[21]
portfolio_assets_rolling_returns_z_scores_50 = asset_return_z_windows[50]
portfolio_assets_rolling_returns_z_scores_200 = asset_return_z_windows[200]
portfolio_assets_rolling_signed_return_z_scores_21 = asset_signed_return_z_windows[21]
portfolio_assets_rolling_signed_return_z_scores_50 = asset_signed_return_z_windows[50]
portfolio_assets_rolling_signed_return_z_scores_200 = asset_signed_return_z_windows[200]
print('[Code Block 9] Calculated asset rolling returns and signed return z-scores for 21/50/200-day windows.')

# Assets / Rolling Sharpe (unsigned + signed)
asset_sharpe_windows = {
    window: _rolling_sharpe(daily_returns, window)
    for window in rolling_windows
}
asset_signed_sharpe_windows = {
    window: _rolling_sharpe(daily_returns_signed, window)
    for window in rolling_windows
}
asset_sharpe_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_sharpe_windows.items()
}
asset_signed_sharpe_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_sharpe_windows.items()
}

portfolio_assets_rolling_sharpe_21 = asset_sharpe_windows[21]
portfolio_assets_rolling_sharpe_50 = asset_sharpe_windows[50]
portfolio_assets_rolling_sharpe_200 = asset_sharpe_windows[200]
portfolio_assets_rolling_signed_sharpe_21 = asset_signed_sharpe_windows[21]
portfolio_assets_rolling_signed_sharpe_50 = asset_signed_sharpe_windows[50]
portfolio_assets_rolling_signed_sharpe_200 = asset_signed_sharpe_windows[200]
portfolio_assets_rolling_sharpe_z_scores_21 = asset_sharpe_z_windows[21]
portfolio_assets_rolling_sharpe_z_scores_50 = asset_sharpe_z_windows[50]
portfolio_assets_rolling_sharpe_z_scores_200 = asset_sharpe_z_windows[200]
portfolio_assets_rolling_signed_sharpe_z_scores_21 = asset_signed_sharpe_z_windows[21]
portfolio_assets_rolling_signed_sharpe_z_scores_50 = asset_signed_sharpe_z_windows[50]
portfolio_assets_rolling_signed_sharpe_z_scores_200 = asset_signed_sharpe_z_windows[200]
print('[Code Block 9] Calculated vectorized asset rolling Sharpe series for 21/50/200-day windows.')

# Assets / Rolling correlation to benchmark (unsigned + signed)
asset_corr_windows = {
    window: daily_returns_for_corr.rolling(window).corr(benchmark_daily_returns_aligned)
    for window in rolling_windows
}
asset_signed_corr_windows = {
    window: daily_returns_signed_for_corr.rolling(window).corr(benchmark_daily_returns_aligned)
    for window in rolling_windows
}
asset_corr_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_corr_windows.items()
}
asset_signed_corr_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_corr_windows.items()
}

portfolio_assets_rolling_correlation_21 = asset_corr_windows[21]
portfolio_assets_rolling_correlation_50 = asset_corr_windows[50]
portfolio_assets_rolling_correlation_200 = asset_corr_windows[200]
portfolio_assets_rolling_signed_correlation_21 = asset_signed_corr_windows[21]
portfolio_assets_rolling_signed_correlation_50 = asset_signed_corr_windows[50]
portfolio_assets_rolling_signed_correlation_200 = asset_signed_corr_windows[200]
portfolio_assets_rolling_correlation_z_scores_21 = asset_corr_z_windows[21]
portfolio_assets_rolling_correlation_z_scores_50 = asset_corr_z_windows[50]
portfolio_assets_rolling_correlation_z_scores_200 = asset_corr_z_windows[200]
portfolio_assets_rolling_signed_correlation_z_scores_21 = asset_signed_corr_z_windows[21]
portfolio_assets_rolling_signed_correlation_z_scores_50 = asset_signed_corr_z_windows[50]
portfolio_assets_rolling_signed_correlation_z_scores_200 = asset_signed_corr_z_windows[200]
print(f'[Code Block 9] Calculated asset rolling correlations to {benchmark_label} for 21/50/200-day windows.')

# Assets / Latest z-scores by metric
latest_assets_return_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_21)
latest_assets_return_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_50)
latest_assets_return_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_200)
latest_assets_sharpe_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_21)
latest_assets_sharpe_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_50)
latest_assets_sharpe_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_200)
latest_assets_signed_sharpe_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_21)
latest_assets_signed_sharpe_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_50)
latest_assets_signed_sharpe_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_200)

# Display labels: add parentheses only for signed series with negative direction
def format_ticker(ticker, sign):
    return f'({ticker})' if sign < 0 else ticker

latest_assets_return_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_21.index]
latest_assets_return_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_50.index]
latest_assets_return_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_200.index]
latest_assets_sharpe_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_21.index]
latest_assets_sharpe_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_50.index]
latest_assets_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_200.index]
latest_assets_signed_sharpe_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_21.index]
latest_assets_signed_sharpe_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_50.index]
latest_assets_signed_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_200.index]

# Portfolio aggregates (signed equal-weight daily rebalance)
portfolio_daily_returns = daily_returns_signed.mean(axis=1)
portfolio_equity = (1 + portfolio_daily_returns).cumprod()
portfolio_return_21 = portfolio_equity.pct_change(21).dropna()
portfolio_return_50 = portfolio_equity.pct_change(50).dropna()
portfolio_return_200 = portfolio_equity.pct_change(200).dropna()
portfolio_rolling_sharpe_21 = _rolling_sharpe(portfolio_daily_returns, 21).dropna()
portfolio_rolling_sharpe_50 = _rolling_sharpe(portfolio_daily_returns, 50).dropna()
portfolio_rolling_sharpe_200 = _rolling_sharpe(portfolio_daily_returns, 200).dropna()

# Benchmark analytics (unsigned benchmark series)
benchmark_equity = (1 + benchmark_daily_returns).cumprod()
benchmark_rolling_sharpe_21 = _rolling_sharpe(benchmark_daily_returns, 21).dropna()
benchmark_rolling_sharpe_50 = _rolling_sharpe(benchmark_daily_returns, 50).dropna()
benchmark_rolling_sharpe_200 = _rolling_sharpe(benchmark_daily_returns, 200).dropna()
portfolio_rolling_signed_correlation_21 = portfolio_daily_returns_for_corr.rolling(21).corr(benchmark_daily_returns_aligned).dropna()
portfolio_rolling_signed_correlation_50 = portfolio_daily_returns_for_corr.rolling(50).corr(benchmark_daily_returns_aligned).dropna()
portfolio_rolling_signed_correlation_200 = portfolio_daily_returns_for_corr.rolling(200).corr(benchmark_daily_returns_aligned).dropna()
print(f'[Code Block 9] Calculated portfolio and benchmark aggregate analytics versus {benchmark_label}.')

calculation_log = [
    '[Code Block 9] Summary:',
    f'- Assets analyzed: {len(portfolio_closing_prices.columns)}',
    f'- Rolling windows: {", ".join(str(window) for window in rolling_windows)} trading days',
    _log_summary('Asset daily returns', daily_returns),
    _log_summary('Signed asset daily returns', daily_returns_signed),
    _log_summary('Signed asset rolling returns (200d)', portfolio_assets_rolling_signed_returns_200),
    _log_summary('Asset rolling Sharpe (200d)', portfolio_assets_rolling_sharpe_200),
    _log_summary(f'Asset rolling correlation to {benchmark_label} (200d)', portfolio_assets_rolling_signed_correlation_200),
    _log_summary('Portfolio equity', portfolio_equity),
    _log_summary('Portfolio rolling Sharpe (200d)', portfolio_rolling_sharpe_200),
    _log_summary(f'Portfolio rolling correlation to {benchmark_label} (200d)', portfolio_rolling_signed_correlation_200),
]
print("\n".join(calculation_log))


In [ ]:
# Code Block 10: Plots
# =========================
# 10) Plots
# =========================

from Quantapp.analytics.series_utils import calculate_zscore

def zscore_for_plot(series):
    cleaned = pd.Series(series).dropna().sort_index()
    if cleaned.empty:
        return pd.Series(dtype=float)
    zscore_series = calculate_zscore(cleaned)
    if zscore_series.isna().all():
        return pd.Series(0.0, index=cleaned.index)
    return zscore_series.dropna()

def add_block16_style_guides(fig):
    fig.add_hrect(y0=-2, y1=-1, fillcolor='rgba(0, 128, 0, 0.30)', line_width=0, layer='below')
    fig.add_hrect(y0=-1, y1=1, fillcolor='rgba(211, 211, 211, 0.18)', line_width=0, layer='below')
    fig.add_hrect(y0=1, y1=2, fillcolor='rgba(180, 0, 0, 0.30)', line_width=0, layer='below')
    fig.add_hline(y=0, line=dict(color='rgba(255, 255, 255, 0.80)', width=1))
    for level in (0.5, 1, 1.5, 2):
        fig.add_hline(y=level, line=dict(color='rgba(220, 220, 220, 0.55)', width=1, dash='dot'))
        fig.add_hline(y=-level, line=dict(color='rgba(220, 220, 220, 0.55)', width=1, dash='dot'))

# 1) Portfolio vs benchmark equity curve (non-z-score)
fig = go.Figure()
fig.add_trace(go.Scatter(x=portfolio_equity.index, y=portfolio_equity, mode='lines', name='Portfolio Equity Curve'))
fig.add_trace(go.Scatter(x=benchmark_equity.index, y=benchmark_equity, mode='lines', name='Benchmark (SPY) Equity Curve'))
fig.update_layout(title='Portfolio vs Benchmark Equity Curve', xaxis_title='Date', yaxis_title='Equity Curve', template='plotly_dark')
fig.show()

# 2) Portfolio vs benchmark rolling sharpe (z-score only)
p21, b21 = zscore_for_plot(portfolio_rolling_sharpe_21), zscore_for_plot(benchmark_rolling_sharpe_21)
p50, b50 = zscore_for_plot(portfolio_rolling_sharpe_50), zscore_for_plot(benchmark_rolling_sharpe_50)
p200, b200 = zscore_for_plot(portfolio_rolling_sharpe_200), zscore_for_plot(benchmark_rolling_sharpe_200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=p21.index, y=p21, mode='lines', name='Portfolio Rolling Sharpe Z-Score (21 days)', visible=False))
fig.add_trace(go.Scatter(x=b21.index, y=b21, mode='lines', name='Benchmark (SPY) Rolling Sharpe Z-Score (21 days)', line=dict(dash='dot'), visible=False))
fig.add_trace(go.Scatter(x=p50.index, y=p50, mode='lines', name='Portfolio Rolling Sharpe Z-Score (50 days)', visible=False))
fig.add_trace(go.Scatter(x=b50.index, y=b50, mode='lines', name='Benchmark (SPY) Rolling Sharpe Z-Score (50 days)', line=dict(dash='dot'), visible=False))
fig.add_trace(go.Scatter(x=p200.index, y=p200, mode='lines', name='Portfolio Rolling Sharpe Z-Score (200 days)'))
fig.add_trace(go.Scatter(x=b200.index, y=b200, mode='lines', name='Benchmark (SPY) Rolling Sharpe Z-Score (200 days)', line=dict(dash='dot')))
add_block16_style_guides(fig)
fig.update_layout(
    title='Portfolio vs Benchmark Rolling Sharpe Z-Score (200 days)',
    xaxis_title='Date',
    yaxis_title='Z-Score',
    template='plotly_dark',
    updatemenus=[
        dict(
            buttons=[
                dict(
                    args=[
                        {'visible': [True, True, False, False, False, False]},
                        {'title': 'Portfolio vs Benchmark Rolling Sharpe Z-Score (21 days)'}
                    ],
                    label='21 Days',
                    method='update'
                ),
                dict(
                    args=[
                        {'visible': [False, False, True, True, False, False]},
                        {'title': 'Portfolio vs Benchmark Rolling Sharpe Z-Score (50 days)'}
                    ],
                    label='50 Days',
                    method='update'
                ),
                dict(
                    args=[
                        {'visible': [False, False, False, False, True, True]},
                        {'title': 'Portfolio vs Benchmark Rolling Sharpe Z-Score (200 days)'}
                    ],
                    label='200 Days',
                    method='update'
                ),
            ],
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            active=2,
            x=0.1,
            xanchor='left',
            y=1.15,
            yanchor='top'
        ),
    ]
)
fig.show()

# 3) Portfolio rolling correlation (non-z-score)
fig = go.Figure()
fig.add_trace(go.Scatter(x=portfolio_rolling_signed_correlation_21.index, y=portfolio_rolling_signed_correlation_21, mode='lines', name='Portfolio Rolling Correlation (21 days)'))
fig.add_trace(go.Scatter(x=portfolio_rolling_signed_correlation_50.index, y=portfolio_rolling_signed_correlation_50, mode='lines', name='Portfolio Rolling Correlation (50 days)', visible=False))
fig.add_trace(go.Scatter(x=portfolio_rolling_signed_correlation_200.index, y=portfolio_rolling_signed_correlation_200, mode='lines', name='Portfolio Rolling Correlation (200 days)', visible=False))
fig.update_layout(
    title='Portfolio Rolling Correlation vs Benchmark (SPY)',
    xaxis_title='Date',
    yaxis_title='Rolling Correlation',
    template='plotly_dark',
    updatemenus=[
        dict(
            buttons=[
                dict(
                    args=[{'visible': [True, False, False]}],
                    label='21 Days',
                    method='update'
                ),
                dict(
                    args=[{'visible': [False, True, False]}],
                    label='50 Days',
                    method='update'
                ),
                dict(
                    args=[{'visible': [False, False, True]}],
                    label='200 Days',
                    method='update'
                ),
            ],
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.1,
            xanchor='left',
            y=1.15,
            yanchor='top'
        ),
    ]
)
fig.add_hline(y=0, line=dict(color='Red', width=2, dash='dash'))
fig.show()


In [ ]:
# Code Block 11: Relative price strength z-score plots
#Relative Price Strength Z-Score Plots
def plot_z_score_diff_dropdown(z_scores_map, title_metric='Return'):
    diff_frames = {}
    global_zmin = np.inf
    global_zmax = -np.inf
    for label, z_scores in z_scores_map.items():
        diff = pd.DataFrame(
            z_scores.values[:, None] - z_scores.values,
            index=z_scores.index,
            columns=z_scores.index
        ).astype(float)
        diff_frames[label] = diff
        global_zmin = min(global_zmin, diff.min().min())
        global_zmax = max(global_zmax, diff.max().max())

    fig = go.Figure()
    labels = list(diff_frames.keys())
    for idx, label in enumerate(labels):
        diff = diff_frames[label]
        fig.add_trace(go.Heatmap(
            z=diff.values,
            x=diff.columns,
            y=diff.index,
            zmin=global_zmin,
            zmax=global_zmax,
            colorscale='RdBu',
            colorbar=dict(title='Z-Score Diff'),
            text=diff.round(2),
            hovertemplate='%{y} vs %{x}<br>Diff: %{z:.2f}<extra></extra>',
            visible=idx == 0
        ))

    buttons = []
    for idx, label in enumerate(labels):
        visibility = [i == idx for i in range(len(labels))]
        buttons.append(
            dict(
                label=f'{label}-Day',
                method='update',
                args=[
                    {"visible": visibility},
                    {"title": f'{title_metric} Pairwise Z-Score Differences ({label}-day)'}
                ]
            )
        )

    fig.update_layout(
        title=f'{title_metric} Pairwise Z-Score Differences ({labels[0]}-day)',
        template='plotly_dark',
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.5,
            xanchor="center",
            y=1.2,
            yanchor="top"
        )],
        height=700
    )
    fig.update_xaxes(title_text='Assets')
    fig.update_yaxes(title_text='Assets')
    fig.show()

from Quantapp.analytics import TimeSeriesAnalytics as Rolling, RiskRelativeAnalytics

rolling = Rolling()
risk_relative_analytics = RiskRelativeAnalytics()

time_frame_map = {
    '21': time_frame_short,
    '50': time_frame_mid,
    '200': time_frame_long,
}

sign_series = net_direction['sign'].reindex(portfolio_closing_prices.columns).fillna(1.0).astype(float)

def format_ticker(ticker, sign):
    return f'({ticker})' if sign < 0 else ticker

def format_snapshot_map(snapshot_map, sign_series):
    formatted = {}
    for label, series in snapshot_map.items():
        formatted_series = series.copy()
        formatted_series.index = [format_ticker(ticker, sign_series.loc[ticker]) for ticker in formatted_series.index]
        formatted[label] = formatted_series
    return formatted

def highlight_signed_bars(index, sign_series, default_color='#636EFA', highlight_color='#F59E0B'):
    colors = []
    for ticker in index:
        raw_ticker = ticker.strip('()') if isinstance(ticker, str) else ticker
        sign_value = sign_series.reindex([raw_ticker]).fillna(1.0).iloc[0]
        colors.append(highlight_color if sign_value < 0 else default_color)
    return colors

benchmark_snapshot = risk_relative_analytics.build_multi_asset_benchmark_snapshot(
    analytics=rolling,
    asset_close=portfolio_closing_prices,
    benchmark_close=benchmark_close,
    time_frame_map=time_frame_map,
    sign_map=sign_series,
)

windows_signed = format_snapshot_map(
    benchmark_snapshot['signed_asset_latest_zscores'],
    sign_series,
)
windows_unsigned = benchmark_snapshot['unsigned_asset_latest_zscores']
windows_benchmark_minus_assets_signed = format_snapshot_map(
    benchmark_snapshot['signed_spread_latest_zscores'],
    sign_series,
)
windows_benchmark_minus_assets_unsigned = benchmark_snapshot['unsigned_spread_latest_zscores']
signed_return_zscore_map = format_snapshot_map(
    benchmark_snapshot['signed_return_latest_zscores'],
    sign_series,
)

fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    subplot_titles=(
        'Asset Sharpe Ratio Z-Scores (Signed)',
        'Asset Sharpe Ratio Z-Scores (Unsigned)',
        f'{benchmark_str} - Asset Sharpe Spread Z-Score (Signed)',
        f'{benchmark_str} - Asset Sharpe Spread Z-Score (Unsigned)',
    )
)

window_labels = list(windows_signed.keys())
for idx, window in enumerate(window_labels):
    z_signed = windows_signed[window]
    z_unsigned = windows_unsigned[window]
    z_diff_signed = windows_benchmark_minus_assets_signed[window]
    z_diff_unsigned = windows_benchmark_minus_assets_unsigned[window]
    unsigned_bar_colors = highlight_signed_bars(z_unsigned.index, sign_series)
    unsigned_spread_bar_colors = highlight_signed_bars(z_diff_unsigned.index, sign_series)

    fig.add_trace(
        go.Bar(x=z_signed.index, y=z_signed.values, name=f'{window}-Day Signed', showlegend=False, visible=idx == 0),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=z_unsigned.index, y=z_unsigned.values, marker_color=unsigned_bar_colors, name=f'{window}-Day Unsigned', showlegend=False, visible=idx == 0),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(x=z_diff_signed.index, y=z_diff_signed.values, name=f'{window}-Day {benchmark_str} - Asset Spread', showlegend=False, visible=idx == 0),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=z_diff_unsigned.index, y=z_diff_unsigned.values, marker_color=unsigned_spread_bar_colors, name=f'{window}-Day {benchmark_str} - Asset Spread', showlegend=False, visible=idx == 0),
        row=2,
        col=2,
    )

for r in [1, 2]:
    for c in [1, 2]:
        fig.add_hline(y=0, line_dash='dash', line_color='red', row=r, col=c)
        fig.add_hline(y=1, line_dash='dot', line_color='green', row=r, col=c)
        fig.add_hline(y=-1, line_dash='dot', line_color='green', row=r, col=c)
        fig.add_hline(y=2, line_dash='dot', line_color='orange', row=r, col=c)
        fig.add_hline(y=-2, line_dash='dot', line_color='orange', row=r, col=c)
        fig.add_hline(y=3, line_dash='dot', line_color='purple', row=r, col=c)
        fig.add_hline(y=-3, line_dash='dot', line_color='purple', row=r, col=c)

        fig.add_hrect(y0=-1, y1=1, fillcolor='lightgreen', opacity=0.3, layer='below', line_width=0, row=r, col=c)
        fig.add_hrect(y0=-2, y1=2, fillcolor='lightyellow', opacity=0.3, layer='below', line_width=0, row=r, col=c)
        fig.add_hrect(y0=-3, y1=3, fillcolor='lightcoral', opacity=0.3, layer='below', line_width=0, row=r, col=c)

buttons = []
for idx, window in enumerate(window_labels):
    visibility = [False] * (4 * len(window_labels))
    base = 4 * idx
    visibility[base] = True
    visibility[base + 1] = True
    visibility[base + 2] = True
    visibility[base + 3] = True

    buttons.append(
        dict(
            label=f'{window}-Day',
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'Sharpe Z-Scores + {benchmark_str} Sharpe Spread Z-Scores - {window}-Day'}
            ]
        )
    )

fig.update_layout(
    title=f'Sharpe Z-Scores + {benchmark_str} Sharpe Spread Z-Scores - 21-Day',
    template='plotly_dark',
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.5,
        xanchor='center',
        y=1.15,
        yanchor='top'
    )],
    height=950
)

fig.update_xaxes(title_text='Assets', row=1, col=1)
fig.update_xaxes(title_text='Assets', row=1, col=2)
fig.update_xaxes(title_text='Assets', row=2, col=1)
fig.update_xaxes(title_text='Assets', row=2, col=2)
fig.update_yaxes(title_text='Z-Score', row=1, col=1)
fig.update_yaxes(title_text='Z-Score', row=1, col=2)
fig.update_yaxes(title_text='Z-Score', row=2, col=1)
fig.update_yaxes(title_text='Z-Score', row=2, col=2)
fig.show()

plot_z_score_diff_dropdown(signed_return_zscore_map, title_metric='Signed Return')







In [ ]:
# Code Block 13: Portfolio allocation optimization helpers
#this tells you to increase capital allocation to the asset. this does not tell you whether or not its a good investment
from scipy.optimize import minimize

#columns are the tickers and rows are the dates
dataframe = raw_prices

def optimal_allocation(dataframe, risk_free_rate=0.0):
    """
    Calculates the optimal allocation of assets in a portfolio to maximize the Sharpe ratio
    using mean-variance optimization.

    :param dataframe: A DataFrame of asset prices (columns=tickers, rows=dates).
    :param risk_free_rate: The risk-free rate for Sharpe ratio calculation.
    :return: A DataFrame with one column 'weight' and rows as asset names, rounded to 2 decimals.
    """
    import numpy as np
    import pandas as pd
    from scipy.optimize import minimize

    # 1) Compute daily returns from price data
    returns_df = dataframe.pct_change().dropna()

    # 2) Calculate average (annualized) returns (252: trading days/year)
    mean_returns = returns_df.mean() * 252

    # 3) Calculate the annualized covariance matrix
    cov_matrix = returns_df.cov() * 252

    # 4) Define the number of assets
    num_assets = len(dataframe.columns)

    # 5) Objective function to maximize the Sharpe ratio => minimize negative Sharpe
    def neg_sharpe(weights):
        ret = np.dot(weights, mean_returns)
        vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(ret - risk_free_rate) / vol

    # 6) Constraint: sum(weights) = 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

    # 7) Bounds: no short selling => each weight in [0,1]
    bounds = tuple((0.0, 1.0) for _ in range(num_assets))

    # 8) Initial guess: equal allocation
    init_guess = [1.0 / num_assets] * num_assets

    # 9) Optimization
    result = minimize(
        neg_sharpe,
        init_guess,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    # 10) Return a DataFrame of weights, rounded to two decimals
    weights_df = pd.DataFrame(result.x, index=dataframe.columns, columns=['weight'])
    weights_df = weights_df.round(2)
    return weights_df

    

def rolling_optimal_allocation(dataframe, window=50, risk_free_rate=0.0):
    """
    Applies the 'optimal_allocation' function over a rolling window.
    Returns a DataFrame where each row corresponds to the weights 
    computed for that window, indexed by the last date in the window.
    """
    import pandas as pd

    # Compute daily returns (drop the first NaN row)
    returns_df = dataframe.pct_change().dropna()

    # List to store each window's optimal weights
    rolling_weights_list = []

    # We only compute weights once we have 'window' days of data
    for i in range(window, len(returns_df) + 1):
        # Price slice for the current rolling window
        slice_data = dataframe.iloc[i - window : i]

        # Reuse the existing 'optimal_allocation' function
        weights_df = optimal_allocation(slice_data, risk_free_rate=risk_free_rate)

        # Append just the weight values (in the same column order as 'dataframe') to the list
        rolling_weights_list.append(weights_df['weight'].values)

    # Create a DataFrame of rolling weights, indexed by the final date in each window
    rolling_index = returns_df.index[window - 1 :]
    rolling_weights_df = pd.DataFrame(
        rolling_weights_list,
        index=rolling_index,
        columns=dataframe.columns
    )

    return rolling_weights_df
'''
def plot_rolling_portfolio_allocation(results):
    """
    Plots the rolling optimal portfolio allocation with annotations for the latest weights.

    Args:
        results (pd.DataFrame): DataFrame containing the portfolio allocation weights.

    Returns:
        None
    """
    import plotly.graph_objects as go

    # Create the Plotly figure
    fig = go.Figure()

    # Add a trace for each asset's weight
    for col in results.columns:
        fig.add_trace(go.Scatter(
            x=results.index,
            y=results[col],
            mode='lines',
            name=col
        ))

    # Add annotations for the latest weight of each asset
    for col in results.columns:
        latest_date = results.index[-1]
        latest_weight = results[col].iloc[-1]
        fig.add_annotation(
            x=latest_date,
            y=latest_weight,
            text=f"{latest_weight:.2f}",
            xanchor='left',
            yanchor='middle',
            showarrow=False,
            font=dict(color=fig.data[results.columns.get_loc(col)].line.color),
            bgcolor="rgba(255,255,255,0.5)"
        )

    # Update layout with title, axis labels, and template
    fig.update_layout(
        title='Rolling Optimal Portfolio Allocation',
        xaxis_title='Date',
        yaxis_title='Weight',
        template='plotly_dark',
        height=600
    )

    # Adjust layout to make space for annotations on the right
    fig.update_layout(
        margin=dict(r=150)
    )

    # Display the plot
    fig.show()
'''

def plot_rolling_portfolio_allocation_stacked(results, smooth_window=5):
    """Plot rolling weights as a stacked area chart with optional smoothing."""
    import plotly.graph_objects as go

    if results.empty:
        print("No rolling allocations were produced. Check window size or input data.")
        return

    smoothed = results.rolling(window=smooth_window, min_periods=1).mean().dropna(how='all')

    fig = go.Figure()
    for col in smoothed.columns:
        fig.add_trace(
            go.Scatter(
                x=smoothed.index,
                y=smoothed[col],
                stackgroup='weights',
                mode='lines',
                name=col,
                #hovertemplate='%{x|%Y-%m-%d}<br>%{y:.2f}<extra>{}</extra>'.format(col)
            )
        )

    fig.update_layout(
        title='Smoothed Rolling Portfolio Weights',
        xaxis_title='Date',
        yaxis_title='Weight',
        template='plotly_dark',
        hovermode='x unified',
        height=600
    )
    fig.show()
    
    
def plot_rolling_portfolio_allocation(results):
    """
    Plots the rolling optimal portfolio allocation with annotations for the latest weights.

    Args:
        results (pd.DataFrame): DataFrame containing the portfolio allocation weights.

    Returns:
        None
    """
    import plotly.graph_objects as go

    # Create the Plotly figure
    fig = go.Figure()

    # Add a trace for each asset's weight
    for col in results.columns:
        fig.add_trace(go.Scatter(
            x=results.index,
            y=results[col],
            mode='lines',
            name=col
        ))

    # Add annotations for the latest weight of each asset
    for col in results.columns:
        latest_date = results.index[-1]
        latest_weight = results[col].iloc[-1]
        fig.add_annotation(
            x=latest_date,
            y=latest_weight,
            text=f"{latest_weight:.2f}",
            xanchor='left',
            yanchor='middle',
            showarrow=False,
            font=dict(color=fig.data[results.columns.get_loc(col)].line.color),
            bgcolor="rgba(255,255,255,0.5)"
        )

    # Update layout with title, axis labels, and template
    fig.update_layout(
        title='Rolling Optimal Portfolio Allocation',
        xaxis_title='Date',
        yaxis_title='Weight',
        template='plotly_dark',
        height=600
    )

    # Adjust layout to make space for annotations on the right
    fig.update_layout(
        margin=dict(r=150)
    )

    # Display the plot
    fig.show()
    
    
benchmark = yf.download('SPY', period=period, interval=interval, auto_adjust=True, progress=False)['Close']
raw_prices = portfolio_closing_prices.copy()

for ticker in raw_prices.columns:
    if ticker in net_direction.index:
        direction = net_direction.loc[ticker, 'directional_value']
        if direction < 0:  # bearish: flip the price series
            first_price = raw_prices[ticker].iloc[0]
            # Flip returns around the first price
            flipped = (first_price ** 2) / raw_prices[ticker]
            
            # Safety: ensure numeric, no inf/NaN
            flipped = flipped.astype(float)
            flipped.replace([np.inf, -np.inf], np.nan, inplace=True)
            flipped.fillna(method='ffill', inplace=True)
            flipped.fillna(method='bfill', inplace=True)
            
            raw_prices[ticker] = flipped
            
#DROP VXX
raw_prices = raw_prices.drop(columns=['VXX'], errors='ignore')


In [ ]:
# Code Block 14: Rolling allocations
#rolling allocations
rolling_optimal_allocations = rolling_optimal_allocation(raw_prices, window=selected_time_frame)
plot_rolling_portfolio_allocation(rolling_optimal_allocations)
plot_rolling_portfolio_allocation_stacked(rolling_optimal_allocations, smooth_window=5)

In [ ]:
# Code Block 15: Archived rolling sharpe snippet
'''
def rolling_sharpe(prices, window=50, risk_free_rate=0.0):
    """ 
    Computes rolling Sharpe ratio for a DataFrame of prices.
    """
    returns = prices.pct_change().dropna()

    if np.isscalar(risk_free_rate):
        excess_returns = returns - risk_free_rate / 252
    else:
        rf_aligned = pd.Series(risk_free_rate, index=returns.index)
        excess_returns = returns.sub(rf_aligned, axis=0)

    rolling_mean = excess_returns.rolling(window).mean()
    rolling_std = excess_returns.rolling(window).std().replace(0, np.nan)
    return (rolling_mean / rolling_std) * np.sqrt(252)

# Compute rolling Sharpe
window = selected_time_frame
rolling_sharpe_df = rolling_sharpe(raw_prices, window=window)

# Convert to long format for Plotly
rolling_sharpe_long = (
    rolling_sharpe_df
    .reset_index()
    .rename(columns={'index': 'Date'})
    .melt(id_vars='Date', var_name='Ticker', value_name='Rolling Sharpe')
    .dropna(subset=['Rolling Sharpe'])
)

fig = px.line(
    rolling_sharpe_long,
    x='Date',
    y='Rolling Sharpe',
    color='Ticker',
    title=f'Rolling {window}-Day Sharpe Ratios of Assets in Options Portfolio',
    labels={'Rolling Sharpe': 'Rolling Sharpe Ratio'},
    template='plotly_dark',
    height=600
)
fig.show()
'''